In [2]:
import time
import pandas as pd
import numpy as np

WINDOW_SEC = 5
DEFAULT_CSV_FS = 256
from core.classifier import load_classifier_model
from core.classifier import predict_window_5s


csv_path = "/home/razzannr/Documents/GitHub/consciousness_app_eeg/muse_af7_zurin_20260508_104304.csv"
signal_column = "AF7"
input_fs = DEFAULT_CSV_FS

model = load_classifier_model()

df = pd.read_csv(csv_path)
signal = df[signal_column].dropna().to_numpy(dtype=float)

window_size = int(input_fs * WINDOW_SEC)

results = []

for i, start in enumerate(range(0, len(signal) - window_size + 1, window_size), start=1):
    raw_window = signal[start:start + window_size]

    t0 = time.perf_counter()

    pred, prob, status, latency, modes, feats, filtered = predict_window_5s(
        raw_window,
        model=model,
        input_fs=input_fs
    )

    t1 = time.perf_counter()

    computation_time = t1 - t0

    results.append({
        "window": i,
        "start_sample": start,
        "end_sample": start + window_size,
        "start_time_sec": start / input_fs,
        "end_time_sec": (start + window_size) / input_fs,
        "prediction": pred,
        "probability": prob,
        "status": status,
        "latency_from_function_sec": latency,
        "computation_time_sec": computation_time
    })

result_df = pd.DataFrame(results)

summary_df = pd.DataFrame({
    "metric": [
        "jumlah_window",
        "mean_time_sec",
        "min_time_sec",
        "max_time_sec",
        "std_time_sec"
    ],
    "value": [
        len(result_df),
        result_df["computation_time_sec"].mean(),
        result_df["computation_time_sec"].min(),
        result_df["computation_time_sec"].max(),
        result_df["computation_time_sec"].std()
    ]
})

result_df.to_csv("zurin_waktu.csv", index=False)
summary_df.to_csv("zurin_ringkasan.csv", index=False)

print(result_df)
print(summary_df)

     window  start_sample  end_sample  start_time_sec  end_time_sec  \
0         1             0        1280             0.0           5.0   
1         2          1280        2560             5.0          10.0   
2         3          2560        3840            10.0          15.0   
3         4          3840        5120            15.0          20.0   
4         5          5120        6400            20.0          25.0   
..      ...           ...         ...             ...           ...   
115     116        147200      148480           575.0         580.0   
116     117        148480      149760           580.0         585.0   
117     118        149760      151040           585.0         590.0   
118     119        151040      152320           590.0         595.0   
119     120        152320      153600           595.0         600.0   

     prediction  probability status  latency_from_function_sec  \
0             0     0.133333     OK                   0.104833   
1             1